# 🔥 Colz GPT — Colab Training Notebook
**Run All → Done. Bas itna hi.**

| Step | Kya hoga |
|------|----------|
| 1 | GPU check |
| 2 | GitHub se code + dataset |
| 3 | transformer / trainer / inference overwrite (speed fixes) |
| 4 | Tokenize + Train (autocast fp16 + torch.compile) |
| 5 | Generation test |
| 6 | brain.pt + vocab.json download |

In [ ]:
# ── CELL 1 : GPU CHECK ────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {name}  |  VRAM: {mem:.1f} GB')
    print(f'   PyTorch: {torch.__version__}')
    print(f'   CUDA   : {torch.version.cuda}')
else:
    print('❌ GPU nahi mila!')
    print('   Runtime → Change runtime type → T4 GPU → Save → Run All')

In [ ]:
# ── CELL 2 : GITHUB CLONE / PULL ─────────────────────────────────────────
import os

GITHUB_REPO = 'https://github.com/niteenmaurya/Colz.git'
FOLDER      = '/content/gpt'

if os.path.exists(FOLDER):
    print('Folder pehle se hai — pulling latest...')
    os.system(f'cd {FOLDER} && git reset --hard HEAD && git pull')
else:
    os.system(f'git clone {GITHUB_REPO} {FOLDER}')

os.chdir(FOLDER)
print(f'\n📁 Workspace : {os.getcwd()}')
print('📁 Files     :', '  '.join(sorted(os.listdir())))

In [ ]:
# ── CELL 3 : WRITE transformer.py ────────────────────────────────────────
transformer_code = '''
import torch
import torch.nn as nn
import math

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"[Transformer] device: {DEVICE}")

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.blocks    = nn.ModuleList([
            _TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.ln_final  = nn.LayerNorm(d_model)
        self.lm_head   = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
        self.to(DEVICE)
        self._last_logits_tensor = None

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, token_ids):
        is_list = isinstance(token_ids, list)
        ids = torch.tensor(token_ids, dtype=torch.long, device=DEVICE) if is_list else token_ids.to(DEVICE)
        if ids.dim() == 1: ids = ids.unsqueeze(0)
        B, T = ids.shape
        if T > self.max_seq_len:
            ids = ids[:, -self.max_seq_len:]; T = self.max_seq_len
        pos    = torch.arange(T, device=DEVICE).unsqueeze(0)
        x      = self.token_emb(ids) + self.pos_emb(pos)
        for blk in self.blocks: x = blk(x)
        x      = self.ln_final(x)
        logits = self.lm_head(x)
        self._last_logits_tensor = logits
        return logits[0].tolist() if is_list else logits

    def backward(self, dlogits): pass

    def zero_grad(self):
        for p in self.parameters():
            if p.grad is not None: p.grad.detach_(); p.grad.zero_()

    def all_params_and_grads(self):
        result = []
        for name, param in self.named_parameters():
            g = param.grad
            gl = g.tolist() if g is not None else [[0.0]*param.shape[-1]]*(param.shape[0] if param.dim()>1 else 1)
            result.append((param.tolist(), gl, name))
        return result

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


class _TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = _CausalSelfAttention(d_model, n_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = _FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class _CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3*d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model,   bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        def reshape(t): return t.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        q, k, v = reshape(q), reshape(k), reshape(v)
        scale  = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(q, k.transpose(-2,-1)) * scale
        mask   = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
        attn   = torch.softmax(scores + mask, dim=-1)
        out    = torch.matmul(attn, v).transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(out)


class _FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
    def forward(self, x): return self.net(x)
'''

with open('transformer.py', 'w', encoding='utf-8') as f:
    f.write(transformer_code)
print('✅ transformer.py written')

In [ ]:
# ── CELL 4 : WRITE trainer.py  (autocast fp16 + GradScaler + batch 256) ──
trainer_code = '''
import math, random, time
import torch
import torch.nn as nn
import torch.optim as optim

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)

class AdamOptimizer:
    def __init__(self, lr=3e-4, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01): self.t=0
    def step(self, params_and_grads): pass

def clip_gradients(p, max_norm=1.0): return max_norm
def cross_entropy_loss(logits, targets): return 0.0, []


class Trainer:
    def __init__(self, model, lr=3e-4, max_norm=1.0, batch_size=256):
        self.model      = model
        self.max_norm   = max_norm
        self.batch_size = batch_size

        self.torch_optimizer = optim.AdamW(
            model.parameters(), lr=lr,
            betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
        )
        self.optimizer    = AdamOptimizer(lr=lr)
        self._use_amp     = (DEVICE.type == "cuda")
        self.scaler       = torch.cuda.amp.GradScaler(enabled=self._use_amp)
        self.loss_history = []
        self._loss_fn     = nn.CrossEntropyLoss(ignore_index=0)

    def train_step(self, batch_windows, seq_len):
        self.torch_optimizer.zero_grad()
        inputs_list, targets_list = [], []

        for token_ids in batch_windows:
            if len(token_ids) < 2: continue
            inp, tgt = token_ids[:-1], token_ids[1:]
            if len(inp) > seq_len: inp, tgt = inp[-seq_len:], tgt[-seq_len:]
            pad = seq_len - len(inp)
            if pad > 0: inp += [0]*pad; tgt += [0]*pad
            inputs_list.append(inp); targets_list.append(tgt)

        if not inputs_list: return 0.0

        inp_t = torch.tensor(inputs_list,  dtype=torch.long, device=DEVICE)
        tgt_t = torch.tensor(targets_list, dtype=torch.long, device=DEVICE)
        B, T  = inp_t.shape

        try:
            with torch.autocast(device_type=DEVICE.type, enabled=self._use_amp):
                pos    = torch.arange(T, device=DEVICE).unsqueeze(0)
                x      = self.model.token_emb(inp_t) + self.model.pos_emb(pos)
                for blk in self.model.blocks: x = blk(x)
                x      = self.model.ln_final(x)
                logits = self.model.lm_head(x)
                loss   = self._loss_fn(logits.view(-1, logits.size(-1)), tgt_t.view(-1))

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.torch_optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_norm)
            self.scaler.step(self.torch_optimizer)
            self.scaler.update()
            return loss.item()
        except Exception as e:
            print(f"⚠️ train_step error: {e}")
            return 0.0

    def train(self, all_token_ids, seq_len, epochs, log_every=5):
        windows = [all_token_ids[s:s+seq_len+1]
                   for s in range(0, len(all_token_ids)-seq_len, seq_len)]
        if not windows: windows = [all_token_ids]

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.torch_optimizer, T_max=epochs, eta_min=1e-5
        )

        print("="*65)
        print(f"🔥  autocast={'ON fp16' if self._use_amp else 'OFF'}  |  device={DEVICE.type.upper()}")
        print(f"👉  batch={self.batch_size}  params={self.model.count_parameters():,}  windows={len(windows)}")
        print("="*65)

        self.model.train()
        t0 = time.time()

        for epoch in range(1, epochs+1):
            te = time.time()
            random.shuffle(windows)
            batches = [windows[i:i+self.batch_size] for i in range(0, len(windows), self.batch_size)]
            el, sr  = 0.0, 0
            for b in batches:
                el += self.train_step(b, seq_len); sr += 1
            avg = el / max(sr, 1)
            self.loss_history.append(avg)
            scheduler.step()
            if epoch % log_every == 0 or epoch == 1:
                ppl = math.exp(min(avg, 20))
                lr  = self.torch_optimizer.param_groups[0]["lr"]
                print(f" 🌟 epoch {epoch:4d}/{epochs} | loss={avg:.4f} | ppl={ppl:.2f} | lr={lr:.2e} | {time.time()-te:.2f}s")

        print(f"✅ Done in {(time.time()-t0)/60:.2f} min")
        self.model.eval()
'''

with open('trainer.py', 'w', encoding='utf-8') as f:
    f.write(trainer_code)
print('✅ trainer.py written  (autocast + GradScaler + batch_size param)')

In [ ]:
# ── CELL 5 : WRITE inference.py  (full GPU, no Python loops) ─────────────
inference_code = '''
import torch
import torch.nn.functional as F

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)

@torch.no_grad()
def _last_logits(model, ids: torch.Tensor) -> torch.Tensor:
    if ids.shape[1] > model.max_seq_len:
        ids = ids[:, -model.max_seq_len:]
    pos    = torch.arange(ids.shape[1], device=DEVICE).unsqueeze(0)
    x      = model.token_emb(ids) + model.pos_emb(pos)
    for blk in model.blocks: x = blk(x)
    return model.lm_head(model.ln_final(x))[0, -1, :]   # (vocab,)

def _top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    if k <= 0 or k >= logits.size(-1): return logits
    thresh = torch.topk(logits, k).values[-1]
    return logits.masked_fill(logits < thresh, float("-inf"))

@torch.no_grad()
def _loop(model, tokenizer, prompt, max_new, temperature, top_k, use_greedy):
    model.eval()
    ids  = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
    generated = []
    for _ in range(max_new):
        logits = _last_logits(model, ids)
        if use_greedy:
            nid = logits.argmax().item()
        else:
            if top_k > 0: logits = _top_k_filter(logits, top_k)
            probs = F.softmax(logits / max(temperature, 1e-6), dim=-1)
            nid   = torch.multinomial(probs, 1).item()
        if nid == tokenizer.EOS_ID: break
        generated.append(nid)
        ids = torch.cat([ids, torch.tensor([[nid]], dtype=torch.long, device=DEVICE)], dim=1)
    return prompt + tokenizer.decode(generated)

def greedy(model, tokenizer, prompt, max_new=100, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tokenizer, prompt, max_new, 1.0, 0, True)

def sample(model, tokenizer, prompt, max_new=100, temperature=0.8, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tokenizer, prompt, max_new, temperature, 0, False)

def top_k_sample(model, tokenizer, prompt, max_new=100, k=10, temperature=0.8, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tokenizer, prompt, max_new, temperature, k, False)

def generate(model, tokenizer, prompt, max_new=100, strategy="top_k", temperature=0.8, k=10, max_seq_len=128):
    s = strategy.lower()
    if s == "greedy":          return greedy(model, tokenizer, prompt, max_new)
    elif s == "sample":        return sample(model, tokenizer, prompt, max_new, temperature)
    elif s in ("top_k","topk"): return top_k_sample(model, tokenizer, prompt, max_new, k, temperature)
    else: raise ValueError(f"Unknown strategy: {strategy}")
'''

with open('inference.py', 'w', encoding='utf-8') as f:
    f.write(inference_code)
print('✅ inference.py written  (full GPU tensor loop, no .tolist() overhead)')

In [ ]:
# ── CELL 6 : DATASET → data.txt ──────────────────────────────────────────
import json, os

SYSTEM_PROMPT = "You are a friendly, helpful, and witty AI assistant."
jsonl_file    = "dataset.jsonl"
corpus_pieces = []

if not os.path.exists(jsonl_file):
    raise FileNotFoundError(f"❌ {jsonl_file} not found in repo. Push it to GitHub first.")

with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        data = json.loads(line)
        corpus_pieces.append(f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n")
        for msg in data.get('messages', []):
            role    = msg.get('role', '').strip().lower()
            content = msg.get('content', '').strip()
            corpus_pieces.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")

CORPUS = ''.join(corpus_pieces)
with open('data.txt', 'w', encoding='utf-8') as f:
    f.write(CORPUS)

print(f'✅ Corpus ready — {len(CORPUS):,} chars  |  saved to data.txt')

In [ ]:
# ── CELL 7 : TOKENIZER ───────────────────────────────────────────────────
import sys, importlib

sys.path.insert(0, os.getcwd())
for mod in ['tokenizer', 'transformer', 'trainer', 'inference']:
    if mod in sys.modules: importlib.reload(sys.modules[mod])

from tokenizer import CharTokenizer

with open('data.txt', 'r', encoding='utf-8') as f:
    CORPUS = f.read()

tok = CharTokenizer()
tok.build_vocab(CORPUS)
tok.save('vocab.json')
all_ids = tok.encode(CORPUS)

print(f'✅ Vocab size : {tok.vocab_size}')
print(f'   Token IDs  : {len(all_ids):,}')

In [ ]:
# ── CELL 8 : MODEL + torch.compile ───────────────────────────────────────
import torch._dynamo
from transformer import GPT

# ─── CONFIG — ek jagah badlo, sab jagah update ──────────────────────────
CFG = dict(
    vocab_size  = tok.vocab_size,
    d_model     = 256,
    n_heads     = 8,
    n_layers    = 6,
    d_ff        = 1024,
    max_seq_len = 128,
)
TRAIN_CFG = dict(
    lr         = 3e-4,
    batch_size = 256,
    epochs     = 2000,
    log_every  = 25,
)
# ─────────────────────────────────────────────────────────────────────────

model = GPT(**CFG)
print(f'✅ Model params : {model.count_parameters():,}')

# torch.compile — ~1.5-2x extra speed on T4 (PyTorch 2.0+)
torch._dynamo.config.suppress_errors = True
model = torch.compile(model)
print('✅ torch.compile active')

In [ ]:
# ── CELL 9 : TRAIN ───────────────────────────────────────────────────────
from trainer import Trainer

trainer = Trainer(
    model,
    lr         = TRAIN_CFG['lr'],
    batch_size = TRAIN_CFG['batch_size'],
)

trainer.train(
    all_ids,
    seq_len   = CFG['max_seq_len'],
    epochs    = TRAIN_CFG['epochs'],
    log_every = TRAIN_CFG['log_every'],
)

In [ ]:
# ── CELL 10 : GENERATION TEST ────────────────────────────────────────────
from inference import generate

SYSTEM_PROMPT = "You are a friendly, helpful, and witty AI assistant."

test_prompts = [
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nHi!<|im_end|>\n<|im_start|>assistant\n",
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nKya chal raha hai?<|im_end|>\n<|im_start|>assistant\n",
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nTell me a joke.<|im_end|>\n<|im_start|>assistant\n",
]

print('='*60)
print('🤖 GENERATION TEST')
print('='*60)

for p in test_prompts:
    out   = generate(model, tok, p, max_new=80, strategy='top_k', k=5, temperature=0.8)
    reply = out[len(p):].strip()
    user  = p.split('<|im_start|>user\n')[-1].split('<|im_end|>')[0].strip()
    print(f'User  : {user}')
    print(f'Bot   : {reply}')
    print('-'*60)

In [ ]:
# ── CELL 11 : SAVE + LOSS CURVE + DOWNLOAD ───────────────────────────────
import pickle
import matplotlib.pyplot as plt
from google.colab import files

# 1. brain.pt
torch.save({
    'model_state_dict' : model.state_dict(),
    'config'           : CFG,
    'loss_history'     : trainer.loss_history,
}, 'brain.pt')

# 2. brain.pkl (backup)
with open('brain.pkl', 'wb') as f:
    pickle.dump(model, f)

# 3. Loss curve
plt.figure(figsize=(10, 4))
plt.plot(trainer.loss_history, color='crimson', linewidth=1.5)
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=120)
plt.show()

print('✅ brain.pt  |  vocab.json  |  loss_curve.png — ready')
print('⬇️  Downloading...')

files.download('brain.pt')
files.download('vocab.json')
files.download('loss_curve.png')

print('🚀 DONE!')